# 07 — Cloud Deployment

The cassette substrate was designed cloud-first. Every read and write routes through `fsspec`, so the same code runs against a local filesystem or an S3 bucket — you swap the URI, nothing else changes.

```python
store = InfonStore("./data/chips", schema_path="schema.json")         # local
store = InfonStore("s3://acme/chips", schema_path="schema.json")      # S3
```

For batch ingestion, `Executor` abstracts away where the SPLADE work runs:

- `SyncExecutor()` — in-process, laptop
- `ProcessExecutor(workers=8)` — multi-core, one machine
- `LambdaExecutor(function_name, region, schema_s3)` — fan out across Lambda

This notebook walks:

1. **Storage layout** — what lives on S3 when you point a store at `s3://bucket/prefix`.
2. **fsspec paths** — the one seam that makes local and cloud identical.
3. **Executor protocol** — how the same ingest call spans from laptop to 100-worker Lambda fan-out.
4. **Container packaging** — why we ship a Lambda container image (not a zip layer) and how to build + push it end-to-end from Python.
5. **Operational runbook** — schema migration in cloud, time-travel in cloud, monitoring hints.

## 1. Storage layout on S3

An `InfonStore` at `s3://acme/chips` lays out five directories under the prefix:

```
s3://acme/chips/
  \u251c\u2500 cassettes/      \u2190 immutable .inf files, one per ingest batch
  \u251c\u2500 index/
  \u2502   \u251c\u2500 by_triple/  \u2190 per-cassette Parquet: (s, p, o, polarity, conf, ts)
  \u2502   \u251c\u2500 by_time/    \u2190 per-cassette Parquet: (ts, s, p, o)
  \u2502   \u2514\u2500 by_anchor/  \u2190 per-cassette Parquet: (anchor, role, s, p, o)
  \u251c\u2500 docs/           \u2190 one .json per ingested doc (idempotency registry)
  \u251c\u2500 _manifest/      \u2190 snapshot chain + HEAD pointer
  \u2514\u2500 _model/         \u2190 optional: trained sheaf GNN (gnn.pt)
```

Everything append-only. A delta ingest writes:
- one new `cassettes/<doc_hash>.inf` per doc,
- three new index shards per cassette,
- one new `_manifest/<snapshot>.json`,
- one overwrite of `_manifest/HEAD` (the only mutation).

No existing bytes are ever rewritten. That's what makes delta ingest on S3 fast: we avoid the read-modify-write cycles that break object-store performance.

## 2. fsspec paths — the one seam

`pyarrow`, `fsspec`, and our own cassette writers all speak URIs. The only difference between local and S3 is the scheme.

Install `s3fs` (`pip install s3fs`) to enable the `s3://` backend. Credentials come from the standard AWS chain — env vars, `~/.aws/credentials`, IAM role on EC2/Lambda. We never ask for keys in code.

Here's the cassette `RangeFetcher` — it has two implementations but callers never pick between them directly; `InfonStore` does it automatically based on the URI scheme.

In [ ]:
from cognition.cassette import LocalFetcher, FsspecFetcher

for cls in (LocalFetcher, FsspecFetcher):
    print(f"{cls.__name__:<16} {cls.__doc__.splitlines()[0] if cls.__doc__ else ''}")

### How the store picks one

Look at `InfonStore._get_fetcher` \u2014 there's one branch on `://` in the root URI. Everything else in the read path uses the returned fetcher without caring which one it is.

In [ ]:
import inspect
from cognition.cassette.store import InfonStore
print(inspect.getsource(InfonStore._get_fetcher))

## 3. Executor protocol — laptop to Lambda

Ingestion fan-out is a separate concern from storage. The `Executor` protocol has one method \u2014 `map(fn, items) -> list[Result]` \u2014 and three implementations:

| Executor | Where work runs | When to pick it |
|---|---|---|
| `SyncExecutor` | In-process, one at a time | Default. Interactive notebooks, <100 docs |
| `ProcessExecutor(workers=N)` | Subprocess pool on one machine | 100\u20131000 docs, CPU-bound extraction |
| `LambdaExecutor(function, region, schema_s3)` | AWS Lambda concurrent invocations | 1000+ docs, need elastic fan-out |

The same `store.ingest(docs, executor=...)` call works with any of them. Swap the executor; the rest is identical.

In [ ]:
from cognition.cassette import SyncExecutor, ProcessExecutor

# SyncExecutor is the default when you don't pass one.
# ProcessExecutor fans out over CPU cores.
ex = ProcessExecutor(workers=4)
print(f"executor: {type(ex).__name__}, workers={ex.workers}")
# Don't actually run a batch here \u2014 the notebook demonstrates the API,
# not the cold-start of 4 SPLADE processes.
ex.close()

### LambdaExecutor

`LambdaExecutor` submits each `IngestBatch` as a `boto3.client('lambda').invoke` with `InvocationType="RequestResponse"`. A `ThreadPoolExecutor` caps in-flight invocations (default 16) so you don't blow through your account-level concurrency limit.

The handler inside the Lambda function is tiny \u2014 it unwraps the event, loads SPLADE from the container's bundled model, runs `extract_infons`, writes cassettes + indexes directly to S3 via fsspec, and returns footer summary JSON. Total: ~100 lines in `cognition/src/cognition/cassette/handler.py`.

```python
from cognition.cassette import LambdaExecutor

ex = LambdaExecutor(
    function_name="cognition-ingest",
    region="us-west-2",
    schema_s3="s3://acme/chips/schema.json",
    max_concurrency=16,
)
store.ingest(docs, executor=ex)
```

`store.ingest` batches docs per worker so the SPLADE cold-start amortizes. Default batch size = `len(docs) / n_workers`.

## 4. Why a container image, not a Lambda layer

PyTorch's CPU wheel is ~1.5GB unzipped. Lambda layers are capped at 250MB unzipped. We tried the layer route first and hit the limit by 7x \u2014 that's documented in `experiments/cassette_lab/probe_lambda_ingest.py`.

Container images have a 10GB limit. The tradeoff:

| Approach | Cold start | Size limit |
|---|---|---|
| Zip layer | ~400ms\u20131s | 250MB unzipped |
| Container image | ~2\u20133s | 10GB |

For ML payloads the container is the only path. The extra cold-start pays for itself on the second invocation (warm container reuse).

`cognition/src/cognition/cassette/lambda_container.py` has the full Python-only pipeline: build the image locally via `docker buildx`, push to ECR (boto3-authenticated), create or update the Lambda function. No shell scripts, no Terraform, no SAM CLI.

In [ ]:
from cognition.cassette import lambda_container as lc

# Four public functions \u2014 each owns one step.
for name in ("build_image", "ensure_ecr_repo", "push_image", "publish_function"):
    fn = getattr(lc, name)
    print(f"{name}{inspect.signature(fn)}")

### Deployment flow

The probe `experiments/cassette_lab/probe_lambda_container.py` has four modes so you can step through without committing to a push:

```
python3 probe_lambda_container.py --dry-run           # write Dockerfile + context, no docker
python3 probe_lambda_container.py --build             # docker buildx, report image size
python3 probe_lambda_container.py --push \\
    --region us-east-1 --repo cognition-ingest        # build + ECR push
python3 probe_lambda_container.py --deploy \\
    --region us-east-1 --repo cognition-ingest \\
    --bucket my-bucket --role-arn arn:...             # full deploy + smoke invoke
```

Default is `--dry-run` so no flag is ever destructive.

## 5. Operational runbook

### Schema migration in the cloud

Migrations are a read-hydrate-rewrite pattern: the store reads every infon under the old schema, applies the `SchemaFunctor`, and writes new cassettes under the new `schema_ref`. On S3 this costs the same bandwidth as a full corpus read plus ~10% for the new writes.

```python
store = InfonStore("s3://acme/chips", schema_path="schema_v1.json")
functor = SchemaFunctor(rename={"toyota_motors": "toyota"},
                         merge={"panasonic_energy": "panasonic"},
                         delete={"tpu"})
print(store.plan_migration(functor, "schema_v2.json").summary())   # preview
store.migrate(functor, "schema_v2.json")                           # commit
```

Old cassettes remain at their existing keys \u2014 time-travel to the pre-migration snapshot is free.

### Time-travel in the cloud

`Manifest.load_at(root, snap_id)` reads one JSON file from `_manifest/<snap>.json` and returns an immutable manifest. Queries against that manifest see only the cassettes that existed at that snapshot. Costs the same as any other manifest read.

```python
from cognition.cassette.index import Manifest
hist = Manifest.list_snapshots("s3://acme/chips")
old  = store.at(hist[-10])          # 10 ingest batches ago
verdict = old.ask(Query().where(subject="toyota", predicate="invest"))
```

### What to monitor

| Metric | Where | What it tells you |
|---|---|---|
| Lambda invocations + duration | CloudWatch | Ingest throughput; cold-start rate |
| S3 GET requests on `cassettes/` | CloudWatch/S3 | Query hydration volume |
| S3 GET requests on `index/by_triple/*.parquet` | CloudWatch/S3 | Query planner scan cost |
| `_manifest/HEAD` object update timestamps | S3 | When the last successful ingest committed |
| `docs/` size | S3 `ls` | How much corpus has been registered |

None of these need a separate observability system \u2014 everything is visible from CloudWatch and S3 metrics.

## 6. A working local-mode demo

This actually runs. The cell below uses the same code path as S3 \u2014 we just point the store at a tempdir so you don't need AWS creds to see it work.

In [ ]:
import json, tempfile, os
from cognition.cassette import InfonStore, Query

SCHEMA = {
    "toyota":  {"type": "actor",    "tokens": ["toyota"]},
    "invest":  {"type": "relation", "tokens": ["invest", "invested"]},
    "batteries": {"type": "feature", "tokens": ["battery", "batteries"]},
}
tmp = tempfile.mkdtemp(prefix="cognition_07_")
sp  = os.path.join(tmp, "schema.json")
with open(sp, "w") as f: json.dump(SCHEMA, f)

# Swap this one string for 's3://your-bucket/chips' to go cloud.
store = InfonStore(os.path.join(tmp, "store"), schema_path=sp)
store.ingest([{"id": "d1", "timestamp": "2026-01-05",
                "text": "Toyota invested in battery technology."}])

print("store layout:")
for d, dirs, files in os.walk(store.root):
    indent = "  " * (d.count(os.sep) - store.root.count(os.sep))
    print(f"{indent}{os.path.basename(d) or store.root}/")
    for fn in files:
        print(f"{indent}  {fn}")

print(f"\nsnapshots: {store.snapshots()}")

---

**What you saw:**

1. The cassette substrate's S3 story is *path-level*: swap a local path for `s3://bucket/prefix` and everything else works.
2. `fsspec` is the single seam. `LocalFetcher` and `FsspecFetcher` are interchangeable; the store picks one from the URI scheme.
3. The `Executor` protocol separates ingestion fan-out from storage. Same `store.ingest(docs, executor=...)` runs on laptop, CPU pool, or Lambda.
4. Lambda deployment is container-image based (not zip-layer) because PyTorch doesn't fit in 250MB. `lambda_container.py` automates the whole build-push-deploy flow in Python.
5. Schema migration and time-travel both work unchanged on S3. Old cassettes are never rewritten \u2014 time-travel snapshots remain resolvable forever.

**Key point:** the `InfonStore` class doesn't know or care where it lives. `ingest()`, `ask()`, `connect()` \u2014 same API, same results, whether backed by a local filesystem or an S3 bucket.

**Next:** [08 — Category Theory](08_category_theory.ipynb) \u2014 theory behind the shipped `SchemaFunctor`.

In [ ]:
import shutil
shutil.rmtree(tmp)
print("Done.")